# Floquet edge--TLS parameter scans, version 3

Production update: completed coupling--frequency, damping, and open-size checkpoints are validated without rerunning their long grids.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-floquet-tls")

from dataclasses import dataclass, replace
from pathlib import Path
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eig, expm
from scipy.signal import find_peaks
from scipy.sparse import csr_matrix, eye, kron
from scipy.sparse.linalg import expm_multiply

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

RUN_G_WINDOW_PILOT = True
RUN_GAMMA_PILOT = True
RUN_SIZE_PILOT = True
RUN_CHANNEL_PILOT = True

RUN_LONG_N6_G_FREQUENCY = False
RUN_LONG_N6_GAMMA = False
RUN_LONG_SIZE_SCALING = False

CELL_TIMEOUT_POLICY_SECONDS = 300

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "cell_timeout_policy_s": CELL_TIMEOUT_POLICY_SECONDS,
    "long_N6_g_frequency": RUN_LONG_N6_G_FREQUENCY,
    "long_N6_gamma": RUN_LONG_N6_GAMMA,
    "long_size_scaling": RUN_LONG_SIZE_SCALING,
})


## 1. Model and scan observables

The chain sites are \(j=0,\ldots,N-1\), and the TLS is site \(d=N\):

\[
H_1=-J\sum_j Z_jZ_{j+1}-\frac{\omega_d}{2}Z_d+H_{ed},
\qquad
H_2=-h\sum_jX_j-\frac{\omega_d}{2}Z_d+H_{ed},
\]

\[
H_{ed}=g(s_+^0\tau_-+s_-^0\tau_+)
=\frac g2(X_0X_d+Y_0Y_d).
\]

The TLS dissipator is \(\gamma_1\mathcal D[\tau_-]+(\gamma_\phi/2)\mathcal D[Z_d]\). We record the edge phasor, transverse TLS phasor, TLS emission proxy, stroboscopic \(\pi\) component, and relative phase for several discard windows from the same trajectory.


In [ ]:
I2 = csr_matrix(np.eye(2, dtype=complex))
X2 = csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex))
Y2 = csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=complex))
Z2 = csr_matrix(np.diag([1.0, -1.0]).astype(complex))
SM2 = csr_matrix(np.array([[0, 1], [0, 0]], dtype=complex))  # |0><1|
SP2 = SM2.getH()


def kron_all(factors):
    out = csr_matrix([[1.0 + 0.0j]])
    for factor in factors:
        out = kron(out, factor, format="csr")
    return out


def site_operator(local_operator, site, n_total):
    return kron_all([
        local_operator if j == site else I2
        for j in range(n_total)
    ])


def operator_lists(n_total):
    return {
        "x": [site_operator(X2, j, n_total) for j in range(n_total)],
        "y": [site_operator(Y2, j, n_total) for j in range(n_total)],
        "z": [site_operator(Z2, j, n_total) for j in range(n_total)],
        "sm": [site_operator(SM2, j, n_total) for j in range(n_total)],
    }


def zero_operator(dimension):
    return csr_matrix((dimension, dimension), dtype=complex)


def computational_ket(bits):
    index = 0
    for bit in bits:
        index = 2 * index + int(bit)
    ket = np.zeros(2 ** len(bits), dtype=complex)
    ket[index] = 1.0
    return ket


def density_vector_from_bits(bits):
    ket = computational_ket(bits)
    rho = np.outer(ket, ket.conjugate())
    return rho.reshape(-1, order="F")


def unvec(vector, dimension):
    return np.asarray(vector).reshape((dimension, dimension), order="F")


def expectation_from_vec(operator, vector, dimension):
    rho = unvec(vector, dimension)
    return float(np.trace(operator.toarray() @ rho).real)


def liouvillian(hamiltonian, collapse_operators):
    dimension = hamiltonian.shape[0]
    identity = eye(dimension, format="csr", dtype=complex)
    generator = -1j * (
        kron(identity, hamiltonian, format="csr")
        - kron(hamiltonian.T, identity, format="csr")
    )
    for collapse in collapse_operators:
        cdc = collapse.getH() @ collapse
        generator = generator + kron(collapse.conjugate(), collapse, format="csr")
        generator = generator - 0.5 * kron(identity, cdc, format="csr")
        generator = generator - 0.5 * kron(cdc.T, identity, format="csr")
    return generator.tocsr()


print("Local-operator algebra checks:")
print("  ||X^2-I|| =", np.linalg.norm((X2 @ X2 - I2).toarray()))
print("  ||[X,Y]-2iZ|| =", np.linalg.norm((X2 @ Y2 - Y2 @ X2 - 2j * Z2).toarray()))
print("  lowering matrix =", SM2.toarray().tolist())


In [ ]:
@dataclass(frozen=True)
class Parameters:
    N: int = 4
    J: float = 1.0
    h: float = 1.0
    alpha_over_pi: float = 0.75
    beta_over_pi: float = 0.90
    g: float = 0.08
    omega_d: float | None = None
    gamma1: float = 0.08
    gamma_phi: float = 0.0
    periods: int = 32
    samples_per_step: int = 2

    @property
    def alpha(self):
        return self.alpha_over_pi * np.pi

    @property
    def beta(self):
        return self.beta_over_pi * np.pi

    @property
    def T1(self):
        return self.beta / (2.0 * self.J)

    @property
    def T2(self):
        return self.alpha / (2.0 * self.h)

    @property
    def T(self):
        return self.T1 + self.T2

    @property
    def Omega(self):
        return 2.0 * np.pi / self.T

    @property
    def tls_frequency(self):
        return self.Omega / 2.0 if self.omega_d is None else self.omega_d


p_fast = Parameters()
print(p_fast)
print({
    "T1": p_fast.T1,
    "T2": p_fast.T2,
    "T": p_fast.T,
    "Omega": p_fast.Omega,
    "Omega_over_2": p_fast.Omega / 2.0,
    "hT2_over_pi": p_fast.h * p_fast.T2 / np.pi,
    "JT1_over_pi": p_fast.J * p_fast.T1 / np.pi,
})


In [ ]:
def signed_pi_component(signal, discard_fraction=0.25):
    signal = np.asarray(signal, dtype=float)
    start = int(np.floor(discard_fraction * len(signal)))
    y = signal[start:]
    return float(np.mean(((-1.0) ** np.arange(len(y))) * y))


_STATIC_MODEL_CACHE = {}


def static_chain_data(N, J, h):
    key = (int(N), float(J), float(h))
    if key not in _STATIC_MODEL_CACHE:
        n_total = N + 1
        ops = operator_lists(n_total)
        dimension = 2 ** n_total
        h_zz = zero_operator(dimension)
        h_x = zero_operator(dimension)
        for j in range(N - 1):
            h_zz = h_zz - J * (ops["z"][j] @ ops["z"][j + 1])
        for j in range(N):
            h_x = h_x - h * ops["x"][j]
        _STATIC_MODEL_CACHE[key] = (ops, h_zz.tocsr(), h_x.tocsr(), dimension)
    return _STATIC_MODEL_CACHE[key]


def build_open_model(parameters):
    N = parameters.N
    d_site = N
    ops, h_zz, h_x, dimension = static_chain_data(N, parameters.J, parameters.h)

    h_d = -0.5 * parameters.tls_frequency * ops["z"][d_site]
    h_ed = parameters.g * (
        ops["sm"][0].getH() @ ops["sm"][d_site]
        + ops["sm"][0] @ ops["sm"][d_site].getH()
    )
    h_xy = 0.5 * parameters.g * (
        ops["x"][0] @ ops["x"][d_site]
        + ops["y"][0] @ ops["y"][d_site]
    )

    collapse_operators = []
    if parameters.gamma1 > 0:
        collapse_operators.append(np.sqrt(parameters.gamma1) * ops["sm"][d_site])
    if parameters.gamma_phi > 0:
        collapse_operators.append(
            np.sqrt(parameters.gamma_phi / 2.0) * ops["z"][d_site]
        )

    h1 = (h_zz + h_d + h_ed).tocsr()
    h2 = (h_x + h_d + h_ed).tocsr()
    identity = eye(dimension, format="csr", dtype=complex)
    tau_z = -ops["z"][d_site]
    p_excited = 0.5 * (identity + tau_z)

    return {
        "dimension": dimension,
        "ops": ops,
        "H1": h1,
        "H2": h2,
        "collapse": collapse_operators,
        "observables": {
            "edge": ops["z"][0],
            "bulk": ops["z"][N // 2],
            "tls_z": tau_z.tocsr(),
            "tls_x": ops["x"][d_site],
            "tls_y": ops["y"][d_site],
            "tls_excited": p_excited.tocsr(),
        },
        "checks": {
            "exchange_identity_error": np.linalg.norm((h_ed - h_xy).toarray()),
            "H1_hermiticity_error": np.linalg.norm((h1 - h1.getH()).toarray()),
            "H2_hermiticity_error": np.linalg.norm((h2 - h2.getH()).toarray()),
        },
    }


model_check = build_open_model(p_fast)
print(model_check["checks"])
assert max(model_check["checks"].values()) < 1e-12


In [ ]:
def simulate_open(parameters, validate=False):
    model = build_open_model(parameters)
    dimension = model["dimension"]
    l1 = liouvillian(model["H1"], model["collapse"])
    l2 = liouvillian(model["H2"], model["collapse"])
    vector = density_vector_from_bits([0] * parameters.N + [0])
    observables = model["observables"]

    continuous_time = []
    continuous = {name: [] for name in observables}
    stroboscopic = {name: [] for name in observables}
    trace_errors = []
    hermiticity_errors = []
    minimum_eigenvalues = []
    current_time = 0.0

    for _ in range(parameters.periods):
        for name, operator in observables.items():
            stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))
        if validate:
            rho = unvec(vector, dimension)
            trace_errors.append(abs(np.trace(rho) - 1.0))
            hermiticity_errors.append(np.linalg.norm(rho - rho.conjugate().T))
            minimum_eigenvalues.append(float(np.min(np.linalg.eigvalsh(rho)).real))

        trajectory_1 = expm_multiply(
            l1, vector, start=0.0, stop=parameters.T1,
            num=parameters.samples_per_step + 1, endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            continuous_time.append(
                current_time + sample * parameters.T1 / parameters.samples_per_step
            )
            for name, operator in observables.items():
                continuous[name].append(
                    expectation_from_vec(operator, trajectory_1[sample], dimension)
                )
        vector = trajectory_1[-1]
        current_time += parameters.T1

        trajectory_2 = expm_multiply(
            l2, vector, start=0.0, stop=parameters.T2,
            num=parameters.samples_per_step + 1, endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            continuous_time.append(
                current_time + sample * parameters.T2 / parameters.samples_per_step
            )
            for name, operator in observables.items():
                continuous[name].append(
                    expectation_from_vec(operator, trajectory_2[sample], dimension)
                )
        vector = trajectory_2[-1]
        current_time += parameters.T2

    for name, operator in observables.items():
        stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))

    if validate:
        rho = unvec(vector, dimension)
        trace_errors.append(abs(np.trace(rho) - 1.0))
        hermiticity_errors.append(np.linalg.norm(rho - rho.conjugate().T))
        minimum_eigenvalues.append(float(np.min(np.linalg.eigvalsh(rho)).real))

    return {
        "parameters": parameters,
        "model": model,
        "time": np.asarray(continuous_time),
        "continuous": {key: np.asarray(value) for key, value in continuous.items()},
        "stroboscopic": {key: np.asarray(value) for key, value in stroboscopic.items()},
        "quality": {
            "max_trace_error": float(np.max(trace_errors)) if trace_errors else np.nan,
            "max_hermiticity_error": float(np.max(hermiticity_errors)) if hermiticity_errors else np.nan,
            "min_eigenvalue": float(np.min(minimum_eigenvalues)) if minimum_eigenvalues else np.nan,
        },
        "final_vector": vector,
    }


def complex_subharmonic_phasor(time_array, signal, omega, discard_time):
    time_array = np.asarray(time_array)
    signal = np.asarray(signal)
    mask = time_array >= discard_time
    t = time_array[mask]
    x = signal[mask]
    if len(t) < 4:
        return np.nan + 1j * np.nan
    x = x - np.mean(x)
    return 2.0 * np.trapezoid(
        x * np.exp(1j * omega * t / 2.0), t
    ) / (t[-1] - t[0])


def run_metrics(run, discard_periods):
    parameters = run["parameters"]
    discard_time = discard_periods * parameters.T
    edge_phasor = complex_subharmonic_phasor(
        run["time"], run["continuous"]["edge"], parameters.Omega, discard_time
    )
    tls_lowering = 0.5 * (
        run["continuous"]["tls_x"] + 1j * run["continuous"]["tls_y"]
    )
    tls_phasor = complex_subharmonic_phasor(
        run["time"], tls_lowering, parameters.Omega, discard_time
    )
    late = run["time"] >= discard_time
    relative_phase = (
        np.angle(tls_phasor / edge_phasor)
        if abs(edge_phasor) > 1e-10 and abs(tls_phasor) > 1e-10
        else np.nan
    )
    return {
        "A_edge": float(abs(edge_phasor)),
        "A_tls_transverse": float(abs(tls_phasor)),
        "phase_tls_minus_edge": float(relative_phase),
        "tls_emission": float(
            parameters.gamma1 * np.mean(run["continuous"]["tls_excited"][late])
        ),
        "Mpi_edge_strobe": abs(
            signed_pi_component(run["stroboscopic"]["edge"], discard_periods / parameters.periods)
        ),
    }


METRIC_KEYS = [
    "A_edge", "A_tls_transverse", "phase_tls_minus_edge",
    "tls_emission", "Mpi_edge_strobe",
]


def metrics_for_windows(run, discard_windows=(8, 20, 40)):
    output = {}
    for discard in discard_windows:
        for key, value in run_metrics(run, discard).items():
            output[f"{key}_d{discard:02d}"] = value
    return output


test_parameters = replace(
    p_fast, N=4, periods=16, omega_d=p_fast.Omega / 2.0
)
test_run = simulate_open(test_parameters, validate=True)
print("sanity quality =", test_run["quality"])
assert test_run["quality"]["max_trace_error"] < 1e-10
assert test_run["quality"]["max_hermiticity_error"] < 1e-10
assert test_run["quality"]["min_eigenvalue"] > -1e-10


In [ ]:
def quadratic_vertex(x_values, y_values, index):
    if index <= 0 or index >= len(x_values) - 1:
        return np.nan, np.nan
    local = slice(index - 1, index + 2)
    a, b, c = np.polyfit(x_values[local], y_values[local], 2)
    if abs(a) < 1e-14:
        return float(x_values[index]), float(y_values[index])
    location = -b / (2.0 * a)
    if not (x_values[index - 1] <= location <= x_values[index + 1]):
        location = x_values[index]
    return float(location), float(np.polyval([a, b, c], location))


def extract_feature_pair(x_values, y_values, kind, center=1.0, search=(0.82, 1.18)):
    x_values = np.asarray(x_values)
    y_values = np.asarray(y_values)
    mask = (x_values >= search[0]) & (x_values <= search[1])
    indices = np.where(mask)[0]
    local = y_values[mask] if kind == "peak" else -y_values[mask]
    dynamic_range = float(np.ptp(local))
    prominence = max(1e-5, 0.04 * dynamic_range)
    candidates_local = find_peaks(local, prominence=prominence)[0]
    candidates = indices[candidates_local]
    left = candidates[x_values[candidates] < center - 1e-6]
    right = candidates[x_values[candidates] > center + 1e-6]
    if len(left) == 0 or len(right) == 0:
        return {
            "resolved": False,
            "left": (np.nan, np.nan),
            "right": (np.nan, np.nan),
            "midpoint": np.nan,
            "half_split_ratio": np.nan,
        }
    if kind == "peak":
        left_index = left[np.argmax(y_values[left])]
        right_index = right[np.argmax(y_values[right])]
    else:
        left_index = left[np.argmin(y_values[left])]
        right_index = right[np.argmin(y_values[right])]
    left_point = quadratic_vertex(x_values, y_values, int(left_index))
    right_point = quadratic_vertex(x_values, y_values, int(right_index))
    return {
        "resolved": True,
        "left": left_point,
        "right": right_point,
        "midpoint": 0.5 * (left_point[0] + right_point[0]),
        "half_split_ratio": 0.5 * (right_point[0] - left_point[0]),
    }


def atomic_savez(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + ".temporary.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def heatmap(axis, x_values, y_values, values, title, colorbar_label):
    image = axis.pcolormesh(x_values, y_values, values, shading="auto")
    axis.set(xlabel=r"$\omega_d/(\Omega/2)$", ylabel=r"$g$", title=title)
    plt.colorbar(image, ax=axis, label=colorbar_label)
    return image


## 2. Matched-window \(g\)--frequency pilot

This enabled pilot uses \(N=4\), 80 periods, four samples per step, seven couplings, and 41 detunings. Metrics are extracted after discarding 8, 20, and 40 periods from the same trajectory. A peak/dip pair is reported only when two local extrema with finite prominence are actually resolved.


In [ ]:
g_values_pilot = np.asarray([0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.16])
ratio_values_pilot = np.linspace(0.78, 1.22, 41)
discard_windows = (8, 20, 40)
pilot_fields = [
    f"{metric}_d{discard:02d}"
    for discard in discard_windows
    for metric in METRIC_KEYS
]
g_window_pilot = {
    field: np.full((len(g_values_pilot), len(ratio_values_pilot)), np.nan)
    for field in pilot_fields
}

if RUN_G_WINDOW_PILOT:
    pilot_start = time.perf_counter()
    for g_index, coupling in enumerate(g_values_pilot):
        row_start = time.perf_counter()
        for ratio_index, ratio in enumerate(ratio_values_pilot):
            parameters = replace(
                p_fast,
                N=4,
                g=float(coupling),
                omega_d=ratio * p_fast.Omega / 2.0,
                periods=80,
                samples_per_step=4,
            )
            run = simulate_open(parameters, validate=False)
            metrics = metrics_for_windows(run, discard_windows)
            for field, value in metrics.items():
                g_window_pilot[field][g_index, ratio_index] = value
        print(
            f"g={coupling:.3f}: {time.perf_counter() - row_start:.1f}s "
            f"({g_index + 1}/{len(g_values_pilot)})"
        )
    print("g-window pilot total runtime (s) =", time.perf_counter() - pilot_start)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    heatmap(
        axes[0], ratio_values_pilot, g_values_pilot,
        g_window_pilot["A_edge_d20"], "Edge response, discard 20T", r"$A_e$",
    )
    heatmap(
        axes[1], ratio_values_pilot, g_values_pilot,
        g_window_pilot["A_tls_transverse_d20"],
        "Transverse TLS response, discard 20T", r"$|c_d|$",
    )
    fig.tight_layout()
    plt.show()
else:
    print("g-window pilot skipped.")


In [ ]:
if RUN_G_WINDOW_PILOT:
    extracted_splittings = {}
    for discard in discard_windows:
        edge_half_splits = []
        tls_half_splits = []
        edge_midpoints = []
        tls_midpoints = []
        for g_index, coupling in enumerate(g_values_pilot):
            edge_pair = extract_feature_pair(
                ratio_values_pilot, g_window_pilot[f"A_edge_d{discard:02d}"][g_index], "dip"
            )
            tls_pair = extract_feature_pair(
                ratio_values_pilot,
                g_window_pilot[f"A_tls_transverse_d{discard:02d}"][g_index], "peak",
            )
            edge_half_splits.append(
                edge_pair["half_split_ratio"] * p_fast.Omega / 2.0
            )
            tls_half_splits.append(
                tls_pair["half_split_ratio"] * p_fast.Omega / 2.0
            )
            edge_midpoints.append(edge_pair["midpoint"])
            tls_midpoints.append(tls_pair["midpoint"])
        extracted_splittings[discard] = {
            "edge": np.asarray(edge_half_splits),
            "tls": np.asarray(tls_half_splits),
            "edge_midpoint": np.asarray(edge_midpoints),
            "tls_midpoint": np.asarray(tls_midpoints),
        }

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
    for discard in discard_windows:
        axes[0].plot(
            g_values_pilot, extracted_splittings[discard]["edge"], "o-",
            label=f"discard {discard}T",
        )
        axes[1].plot(
            g_values_pilot, extracted_splittings[discard]["tls"], "o-",
            label=f"discard {discard}T",
        )
    for axis in axes:
        axis.plot(g_values_pilot, g_values_pilot, "k--", lw=1, label=r"bare $g$")
        axis.set(xlabel=r"bare coupling $g$", ylabel="half separation [frequency units]")
        axis.legend(fontsize=8)
    axes[0].set_title("Edge-dip separation and window sensitivity")
    axes[1].set_title("TLS-peak separation and window sensitivity")
    fig.tight_layout()
    plt.show()

    print("discard-20T extracted half separations:")
    for index, coupling in enumerate(g_values_pilot):
        print({
            "g": coupling,
            "edge_half_split": extracted_splittings[20]["edge"][index],
            "tls_half_split": extracted_splittings[20]["tls"][index],
            "edge_midpoint": extracted_splittings[20]["edge_midpoint"][index],
            "tls_midpoint": extracted_splittings[20]["tls_midpoint"][index],
        })


## 3. Coarse \(N=6\) damping pilot

The existing \(N=4\) scan already shows non-monotonic loading. This enabled 13-point \(N=6\) pilot checks whether the same crossover survives at the production size. It is not a replacement for the disabled 31-point production scan.


In [ ]:
gamma_values_pilot = np.geomspace(0.005, 3.0, 13)
gamma_pilot = {
    field: np.full(len(gamma_values_pilot), np.nan)
    for field in pilot_fields
}

if RUN_GAMMA_PILOT:
    gamma_start = time.perf_counter()
    for index, gamma1 in enumerate(gamma_values_pilot):
        parameters = replace(
            p_fast,
            N=6,
            g=0.08,
            gamma1=float(gamma1),
            omega_d=p_fast.Omega / 2.0,
            periods=80,
            samples_per_step=2,
        )
        run = simulate_open(parameters, validate=False)
        metrics = metrics_for_windows(run, discard_windows)
        for field, value in metrics.items():
            gamma_pilot[field][index] = value
    print("N=6 gamma pilot runtime (s) =", time.perf_counter() - gamma_start)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for discard in discard_windows:
        axes[0].semilogx(
            gamma_values_pilot, gamma_pilot[f"A_edge_d{discard:02d}"], "o-",
            label=f"discard {discard}T",
        )
        axes[1].loglog(
            gamma_values_pilot, gamma_pilot[f"tls_emission_d{discard:02d}"], "o-",
            label=f"discard {discard}T",
        )
    axes[0].set(
        xlabel=r"$\gamma_1$", ylabel=r"$A_e$",
        title="N=6 edge response versus TLS damping",
    )
    axes[1].set(
        xlabel=r"$\gamma_1$", ylabel=r"$\gamma_1\overline{p_e}$",
        title="N=6 non-monotonic TLS loading",
    )
    for axis in axes:
        axis.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    for discard in discard_windows:
        peak_index = int(np.argmax(gamma_pilot[f"tls_emission_d{discard:02d}"]))
        print(
            f"discard {discard}T: maximum emission at gamma1 = "
            f"{gamma_values_pilot[peak_index]:.6f}"
        )
else:
    print("N=6 gamma pilot skipped.")


## 4. Short open-system size diagnostic

This enabled calculation compares \(N=3,4,5,6\) at nominal resonance and at the two production edge-dip frequencies. Four sizes are not a thermodynamic scaling law; the purpose is to detect gross size drift before committing to larger local calculations.


In [ ]:
size_values_pilot = np.asarray([3, 4, 5, 6])
size_ratio_values = np.asarray([0.945, 1.000, 1.0733333333333333])
size_pilot = {
    "A_edge": np.full((len(size_values_pilot), len(size_ratio_values)), np.nan),
    "A_tls_transverse": np.full((len(size_values_pilot), len(size_ratio_values)), np.nan),
    "tls_emission": np.full((len(size_values_pilot), len(size_ratio_values)), np.nan),
}

if RUN_SIZE_PILOT:
    size_start = time.perf_counter()
    for n_index, chain_size in enumerate(size_values_pilot):
        for ratio_index, ratio in enumerate(size_ratio_values):
            parameters = replace(
                p_fast,
                N=int(chain_size),
                g=0.08,
                gamma1=0.08,
                omega_d=ratio * p_fast.Omega / 2.0,
                periods=80,
                samples_per_step=2,
            )
            run = simulate_open(parameters, validate=False)
            metrics = run_metrics(run, discard_periods=20)
            for key in size_pilot:
                size_pilot[key][n_index, ratio_index] = metrics[key]
    print("size pilot runtime (s) =", time.perf_counter() - size_start)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.0), sharex=True)
    for ratio_index, ratio in enumerate(size_ratio_values):
        label = fr"$\omega_d/(\Omega/2)={ratio:.3f}$"
        axes[0].plot(size_values_pilot, size_pilot["A_edge"][:, ratio_index], "o-", label=label)
        axes[1].plot(
            size_values_pilot, size_pilot["A_tls_transverse"][:, ratio_index], "o-", label=label
        )
        axes[2].plot(size_values_pilot, size_pilot["tls_emission"][:, ratio_index], "o-", label=label)
    axes[0].set(ylabel=r"$A_e$", title="Edge response")
    axes[1].set(ylabel=r"$|c_d|$", title="Transverse TLS response")
    axes[2].set(ylabel=r"$\gamma_1\overline{p_e}$", title="TLS emission")
    for axis in axes:
        axis.set_xlabel("chain length N")
        axis.set_xticks(size_values_pilot)
        axis.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
else:
    print("size pilot skipped.")


## 5. Small-channel detuning pilot

For \(N=3\), the complete one-period quantum channel is diagonalized at three couplings and 17 detunings. All modes within a phase window around \(\pi\) are ranked by their contribution to the selected initial state and left-edge readout. The scatter plots show channel phase offsets, decay moduli, and visibility without prematurely reducing the spectrum to one fitted branch.


In [ ]:
channel_g_values = np.asarray([0.04, 0.08, 0.12])
channel_ratio_values = np.linspace(0.85, 1.15, 17)
channel_records = []


def visible_channel_modes(
    parameters, phase_window=0.55, modes_to_keep=8, return_full=False
):
    model = build_open_model(parameters)
    l1 = liouvillian(model["H1"], model["collapse"]).toarray()
    l2 = liouvillian(model["H2"], model["collapse"]).toarray()
    channel = expm(l2 * parameters.T2) @ expm(l1 * parameters.T1)
    eigenvalues, right_vectors = np.linalg.eig(channel)
    initial_vector = density_vector_from_bits([0] * parameters.N + [0])
    coefficients = np.linalg.solve(right_vectors, initial_vector)
    edge_vector = model["observables"]["edge"].toarray().reshape(-1, order="F")
    readout = edge_vector.conjugate() @ right_vectors
    visibility = np.abs(readout * coefficients)
    phase_offset = np.angle(np.exp(1j * (np.angle(eigenvalues) - np.pi)))
    phase_offset_over_T = phase_offset / parameters.T
    if return_full:
        return {
            "eigenvalues": eigenvalues,
            "right_vectors": right_vectors,
            "visibility": visibility,
            "phase_offset_over_T": phase_offset_over_T,
        }
    candidates = np.where(np.abs(phase_offset) < phase_window)[0]
    order = candidates[np.argsort(visibility[candidates])[::-1]]
    selected = order[:modes_to_keep]
    return [
        {
            "phase_offset_over_T": float(phase_offset_over_T[index]),
            "modulus": float(abs(eigenvalues[index])),
            "visibility": float(visibility[index]),
            "eigenvalue": eigenvalues[index],
        }
        for index in selected
    ]


if RUN_CHANNEL_PILOT:
    channel_start = time.perf_counter()
    for coupling in channel_g_values:
        for ratio in channel_ratio_values:
            parameters = replace(
                p_fast,
                N=3,
                g=float(coupling),
                omega_d=ratio * p_fast.Omega / 2.0,
            )
            for mode in visible_channel_modes(parameters):
                channel_records.append({
                    "g": coupling,
                    "omega_ratio": ratio,
                    **mode,
                })
    print("channel pilot runtime (s) =", time.perf_counter() - channel_start)

    fig, axes = plt.subplots(1, len(channel_g_values), figsize=(15, 4.3), sharey=True)
    scatter = None
    for axis, coupling in zip(axes, channel_g_values):
        records = [row for row in channel_records if np.isclose(row["g"], coupling)]
        visibility_values = np.asarray([row["visibility"] for row in records])
        marker_sizes = 10.0 + 170.0 * visibility_values / max(np.max(visibility_values), 1e-12)
        scatter = axis.scatter(
            [row["omega_ratio"] for row in records],
            [row["phase_offset_over_T"] for row in records],
            s=marker_sizes,
            c=[row["modulus"] for row in records],
            cmap="viridis",
            vmin=0.80,
            vmax=1.00,
            alpha=0.75,
        )
        axis.axhline(0.0, color="k", ls=":", lw=1)
        axis.set(
            xlabel=r"$\omega_d/(\Omega/2)$",
            title=fr"$g={coupling:.2f}$",
        )
    axes[0].set_ylabel(r"channel phase offset from $\pi$, divided by $T$")
    fig.colorbar(scatter, ax=axes, label=r"$|\lambda|$", shrink=0.85)
    fig.suptitle("Visible pi-sector channel modes; marker size is initial/readout visibility")
    plt.show()

    for coupling in channel_g_values:
        central = [
            row for row in channel_records
            if np.isclose(row["g"], coupling) and np.isclose(row["omega_ratio"], 1.0)
        ]
        print("g =", coupling, "central visible modes =", central[:4])

    channel_scaling_g = g_values_pilot.copy()
    channel_scaling_spectra = {}
    for coupling in channel_scaling_g:
        parameters = replace(
            p_fast,
            N=3,
            g=float(coupling),
            omega_d=p_fast.Omega / 2.0,
        )
        channel_scaling_spectra[coupling] = visible_channel_modes(
            parameters, return_full=True
        )

    # Start from the most visible conjugate pair at the largest coupling and
    # track downward by right-eigenvector overlap. This avoids switching
    # branches merely because their readout visibility crosses.
    descending_g = channel_scaling_g[::-1]
    tracked_branches = {1: [], -1: []}
    largest_spectrum = channel_scaling_spectra[descending_g[0]]
    for sign in [1, -1]:
        offset = largest_spectrum["phase_offset_over_T"]
        candidates = np.where((sign * offset > 0.005) & (sign * offset < 0.20))[0]
        selected = candidates[
            np.argmax(largest_spectrum["visibility"][candidates])
        ]
        tracked_branches[sign].append({
            "g": descending_g[0],
            "index": int(selected),
            "vector": largest_spectrum["right_vectors"][:, selected],
            "offset": float(offset[selected]),
            "visibility": float(largest_spectrum["visibility"][selected]),
            "overlap": 1.0,
        })

    for coupling in descending_g[1:]:
        spectrum = channel_scaling_spectra[coupling]
        for sign in [1, -1]:
            previous_vector = tracked_branches[sign][-1]["vector"]
            offset = spectrum["phase_offset_over_T"]
            candidates = np.where((sign * offset > 0.0) & (sign * offset < 0.20))[0]
            overlaps = np.abs(
                previous_vector.conjugate() @ spectrum["right_vectors"][:, candidates]
            )
            selected = candidates[np.argmax(overlaps)]
            tracked_branches[sign].append({
                "g": coupling,
                "index": int(selected),
                "vector": spectrum["right_vectors"][:, selected],
                "offset": float(offset[selected]),
                "visibility": float(spectrum["visibility"][selected]),
                "overlap": float(np.max(overlaps)),
            })

    channel_half_splitting = np.full_like(channel_scaling_g, np.nan, dtype=float)
    channel_branch_visibility = np.full_like(channel_scaling_g, np.nan, dtype=float)
    channel_branch_overlap = np.full_like(channel_scaling_g, np.nan, dtype=float)
    for index, coupling in enumerate(channel_scaling_g):
        positive = next(row for row in tracked_branches[1] if np.isclose(row["g"], coupling))
        negative = next(row for row in tracked_branches[-1] if np.isclose(row["g"], coupling))
        channel_half_splitting[index] = 0.5 * (positive["offset"] - negative["offset"])
        channel_branch_visibility[index] = min(
            positive["visibility"], negative["visibility"]
        )
        channel_branch_overlap[index] = min(positive["overlap"], negative["overlap"])

    resolved_channel = (
        np.isfinite(channel_half_splitting)
        & (channel_half_splitting > 0.005)
        & (channel_scaling_g >= 0.06)
    )
    channel_linear_coefficients = np.polyfit(
        channel_scaling_g[resolved_channel],
        channel_half_splitting[resolved_channel],
        1,
    )
    channel_linear_prediction = np.polyval(
        channel_linear_coefficients, channel_scaling_g[resolved_channel]
    )
    channel_linear_r2 = 1.0 - np.sum(
        (channel_half_splitting[resolved_channel] - channel_linear_prediction) ** 2
    ) / np.sum(
        (channel_half_splitting[resolved_channel]
         - np.mean(channel_half_splitting[resolved_channel])) ** 2
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(
        channel_scaling_g, channel_half_splitting, "o-",
        label="overlap-tracked channel pair",
    )
    dense_g = np.linspace(channel_scaling_g[resolved_channel].min(), channel_scaling_g.max(), 300)
    axes[0].plot(
        dense_g,
        np.polyval(channel_linear_coefficients, dense_g),
        "--",
        label=fr"linear guide for $g\geq0.06$, $R^2={channel_linear_r2:.3f}$",
    )
    axes[0].set(
        xlabel=r"bare coupling $g$",
        ylabel=r"central channel half phase splitting $\delta\phi/T$",
        title="Tracked channel phase separation",
    )
    axes[0].legend(fontsize=8)

    axes[1].semilogy(
        channel_scaling_g, channel_branch_visibility, "o-", label="minimum branch visibility"
    )
    axes[1].plot(
        channel_scaling_g, channel_branch_overlap, "s--", label="minimum step overlap"
    )
    axes[1].set(
        xlabel=r"bare coupling $g$", ylabel="tracking diagnostic",
        title="The tracked branch becomes dark at weak coupling",
    )
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    print("overlap-tracked channel half splittings =", dict(zip(channel_scaling_g, channel_half_splitting)))
    print("tracked branch visibilities =", dict(zip(channel_scaling_g, channel_branch_visibility)))
    print("minimum step overlaps =", dict(zip(channel_scaling_g, channel_branch_overlap)))
    print("resolved-regime linear coefficients [slope, intercept] =", channel_linear_coefficients)
    print("resolved-regime linear R^2 =", channel_linear_r2)
else:
    print("channel pilot skipped.")


## 6. Disabled production scans for local execution

Set the relevant flag at the top to `True`. Each long scan saves after every completed point and resumes from the same checkpoint. Do not change a grid while reusing its checkpoint.

### Production \(g\)--frequency grid

- \(N=6\), 80 periods, four samples per step;
- \(g=0.02,0.04,0.06,0.08,0.10,0.12,0.16\);
- 61 detunings from 0.75 to 1.25;
- discard windows 8, 20, and 40 periods.

### Production damping grid

- \(N=6\), 31 logarithmic \(\gamma_1\) values from 0.005 to 3;
- nominal resonance and the same three discard windows.


In [ ]:
def initialize_grid_checkpoint(path, g_values, ratio_values, windows):
    path = Path(path)
    fields = [
        f"{metric}_d{discard:02d}"
        for discard in windows
        for metric in METRIC_KEYS
    ]
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            data = {key: np.asarray(saved[key]) for key in saved.files}
        assert np.allclose(data["g_values"], g_values)
        assert np.allclose(data["omega_ratios"], ratio_values)
        assert np.array_equal(data["discard_windows"], np.asarray(windows))
        return data
    data = {
        "g_values": np.asarray(g_values),
        "omega_ratios": np.asarray(ratio_values),
        "discard_windows": np.asarray(windows),
        "completed": np.zeros((len(g_values), len(ratio_values)), dtype=bool),
        "metadata_N": np.asarray(6),
        "metadata_periods": np.asarray(80),
        "metadata_samples_per_step": np.asarray(4),
        "metadata_gamma1": np.asarray(0.08),
        "metadata_Omega": np.asarray(p_fast.Omega),
        "metadata_T": np.asarray(p_fast.T),
    }
    for field in fields:
        data[field] = np.full((len(g_values), len(ratio_values)), np.nan)
    return data


def run_long_N6_g_frequency_scan():
    path = Path("floquet_tls_N6_g_frequency_checkpoint.npz")
    g_values = np.asarray([0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.16])
    ratio_values = np.linspace(0.75, 1.25, 61)
    windows = (8, 20, 40)
    data = initialize_grid_checkpoint(path, g_values, ratio_values, windows)
    total_points = data["completed"].size
    for g_index, coupling in enumerate(g_values):
        for ratio_index, ratio in enumerate(ratio_values):
            if data["completed"][g_index, ratio_index]:
                continue
            parameters = replace(
                p_fast,
                N=6,
                g=float(coupling),
                gamma1=0.08,
                omega_d=ratio * p_fast.Omega / 2.0,
                periods=80,
                samples_per_step=4,
            )
            point_start = time.perf_counter()
            run = simulate_open(parameters, validate=False)
            metrics = metrics_for_windows(run, windows)
            for field, value in metrics.items():
                data[field][g_index, ratio_index] = value
            data["completed"][g_index, ratio_index] = True
            atomic_savez(path, **data)
            completed = int(np.count_nonzero(data["completed"]))
            print(
                f"{completed:03d}/{total_points}  g={coupling:.3f}  ratio={ratio:.4f}  "
                f"runtime={time.perf_counter() - point_start:.1f}s"
            )
    return data


if RUN_LONG_N6_G_FREQUENCY:
    long_N6_g_frequency_data = run_long_N6_g_frequency_scan()
else:
    print("Long N=6 g-frequency scan skipped.")


In [ ]:
def initialize_gamma_checkpoint(path, gamma_values, windows):
    path = Path(path)
    fields = [
        f"{metric}_d{discard:02d}"
        for discard in windows
        for metric in METRIC_KEYS
    ]
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            data = {key: np.asarray(saved[key]) for key in saved.files}
        assert np.allclose(data["gamma_values"], gamma_values)
        assert np.array_equal(data["discard_windows"], np.asarray(windows))
        return data
    data = {
        "gamma_values": np.asarray(gamma_values),
        "discard_windows": np.asarray(windows),
        "completed": np.zeros(len(gamma_values), dtype=bool),
        "metadata_N": np.asarray(6),
        "metadata_periods": np.asarray(80),
        "metadata_samples_per_step": np.asarray(4),
        "metadata_g": np.asarray(0.08),
        "metadata_omega_ratio": np.asarray(1.0),
        "metadata_Omega": np.asarray(p_fast.Omega),
        "metadata_T": np.asarray(p_fast.T),
    }
    for field in fields:
        data[field] = np.full(len(gamma_values), np.nan)
    return data


def run_long_N6_gamma_scan():
    path = Path("floquet_tls_N6_gamma_checkpoint.npz")
    gamma_values = np.geomspace(0.005, 3.0, 31)
    windows = (8, 20, 40)
    data = initialize_gamma_checkpoint(path, gamma_values, windows)
    for index, gamma1 in enumerate(gamma_values):
        if data["completed"][index]:
            continue
        parameters = replace(
            p_fast,
            N=6,
            g=0.08,
            gamma1=float(gamma1),
            omega_d=p_fast.Omega / 2.0,
            periods=80,
            samples_per_step=4,
        )
        point_start = time.perf_counter()
        run = simulate_open(parameters, validate=False)
        metrics = metrics_for_windows(run, windows)
        for field, value in metrics.items():
            data[field][index] = value
        data["completed"][index] = True
        atomic_savez(path, **data)
        print(
            f"{np.count_nonzero(data['completed']):02d}/{len(gamma_values)}  "
            f"gamma1={gamma1:.6f}  runtime={time.perf_counter() - point_start:.1f}s"
        )
    return data


if RUN_LONG_N6_GAMMA:
    long_N6_gamma_data = run_long_N6_gamma_scan()
else:
    print("Long N=6 gamma scan skipped.")


In [ ]:
def run_long_size_scaling():
    path = Path("floquet_tls_open_size_checkpoint.npz")
    size_values = np.asarray([3, 4, 5, 6, 7])
    ratio_values = np.asarray([0.945, 1.000, 1.0733333333333333])
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            data = {key: np.asarray(saved[key]) for key in saved.files}
        assert np.array_equal(data["N_values"], size_values)
        assert np.allclose(data["omega_ratios"], ratio_values)
    else:
        data = {
            "N_values": size_values,
            "omega_ratios": ratio_values,
            "completed": np.zeros((len(size_values), len(ratio_values)), dtype=bool),
            "A_edge_d20": np.full((len(size_values), len(ratio_values)), np.nan),
            "A_tls_transverse_d20": np.full((len(size_values), len(ratio_values)), np.nan),
            "tls_emission_d20": np.full((len(size_values), len(ratio_values)), np.nan),
        }
    for n_index, chain_size in enumerate(size_values):
        for ratio_index, ratio in enumerate(ratio_values):
            if data["completed"][n_index, ratio_index]:
                continue
            parameters = replace(
                p_fast,
                N=int(chain_size),
                g=0.08,
                gamma1=0.08,
                omega_d=ratio * p_fast.Omega / 2.0,
                periods=80,
                samples_per_step=2,
            )
            point_start = time.perf_counter()
            run = simulate_open(parameters, validate=False)
            metrics = run_metrics(run, 20)
            for key in ["A_edge", "A_tls_transverse", "tls_emission"]:
                data[f"{key}_d20"][n_index, ratio_index] = metrics[key]
            data["completed"][n_index, ratio_index] = True
            atomic_savez(path, **data)
            print(
                f"N={chain_size}, ratio={ratio:.4f}, "
                f"runtime={time.perf_counter() - point_start:.1f}s"
            )
    return data


if RUN_LONG_SIZE_SCALING:
    long_size_data = run_long_size_scaling()
else:
    print("Long open-system size scaling skipped.")


## 7. Reading completed local checkpoints

After a local long scan finishes, rerun this cell. It verifies completeness before plotting. Partial checkpoints remain usable for progress inspection, but no scaling fit is performed until every requested point is present.


In [ ]:
# Completed N=6 g-frequency production checkpoint
checkpoint_candidates = [
    Path("floquet_tls_N6_g_frequency_checkpoint.npz"),
    Path("upload/floquet_tls_N6_g_frequency_checkpoint.npz"),
]
long_g_path = next((path for path in checkpoint_candidates if path.exists()), None)
if long_g_path is None:
    raise FileNotFoundError(
        "Place floquet_tls_N6_g_frequency_checkpoint.npz beside this notebook."
    )

with np.load(long_g_path, allow_pickle=False) as saved:
    loaded_long_g = {key: np.asarray(saved[key]) for key in saved.files}

required_metadata = {
    "metadata_N": 6,
    "metadata_periods": 80,
    "metadata_samples_per_step": 4,
    "metadata_gamma1": 0.08,
}
for key, expected in required_metadata.items():
    actual = loaded_long_g[key].item()
    if not np.isclose(actual, expected):
        raise ValueError(f"{key}: expected {expected}, found {actual}")

g_production = loaded_long_g["g_values"]
ratio_production = loaded_long_g["omega_ratios"]
windows_production = loaded_long_g["discard_windows"].astype(int)
completed = loaded_long_g["completed"]
if completed.shape != (len(g_production), len(ratio_production)):
    raise ValueError("checkpoint completion mask has the wrong shape")
if not completed.all():
    raise RuntimeError(f"checkpoint is only {completed.mean():.1%} complete")

metric_names = [
    "A_edge", "A_tls_transverse", "phase_tls_minus_edge",
    "tls_emission", "Mpi_edge_strobe",
]
for discard in windows_production:
    for metric in metric_names:
        values = loaded_long_g[f"{metric}_d{discard:02d}"]
        if values.shape != completed.shape or not np.isfinite(values).all():
            raise ValueError(f"invalid production array: {metric}, discard={discard}")


def production_tls_peak_pair(x_values, y_values, center=1.0):
    """Return an observed inner peak pair; never manufacture missing peaks."""
    dynamic_range = float(np.ptp(y_values))
    peaks, properties = find_peaks(
        y_values,
        prominence=max(1e-8, 0.04 * dynamic_range),
        distance=3,
    )
    left = [index for index in peaks if 0.86 <= x_values[index] < center]
    right = [index for index in peaks if center < x_values[index] <= 1.24]
    if not left or not right:
        return None
    # The inner pair follows the central resonance. A separate feature near
    # ratio 0.817 is excluded explicitly by the lower search bound.
    left_index = max(left, key=lambda index: x_values[index])
    right_index = min(right, key=lambda index: x_values[index])
    left_point = quadratic_vertex(x_values, y_values, left_index)
    right_point = quadratic_vertex(x_values, y_values, right_index)
    return {
        "left": left_point[0],
        "right": right_point[0],
        "left_height": left_point[1],
        "right_height": right_point[1],
        "midpoint": 0.5 * (left_point[0] + right_point[0]),
        "half_split_ratio": 0.5 * (right_point[0] - left_point[0]),
    }


fig, axes = plt.subplots(3, 3, figsize=(13.5, 9.0), sharex=True)
heatmap_metrics = [
    ("A_edge", r"$A_{edge}$"),
    ("A_tls_transverse", r"$A_{TLS}^{\perp}$"),
    ("tls_emission", r"$\gamma_1\langle n_d\rangle$"),
]
for row, discard in enumerate(windows_production):
    for column, (metric, label) in enumerate(heatmap_metrics):
        heatmap(
            axes[row, column], ratio_production, g_production,
            loaded_long_g[f"{metric}_d{discard:02d}"],
            f"{label}, discard={discard}", label,
        )
        axes[row, column].axvline(1.0, color="w", ls="--", lw=0.8, alpha=0.7)
fig.suptitle("Completed N=6 frequency scan: three matched observation windows", y=1.01)
fig.tight_layout()
plt.show()

production_pairs = {}
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(g_production)))
fig, axes = plt.subplots(1, len(windows_production), figsize=(15, 4.3), sharex=True, sharey=True)
for axis, discard in zip(axes, windows_production):
    tls_values = loaded_long_g[f"A_tls_transverse_d{discard:02d}"]
    for index, (coupling, color) in enumerate(zip(g_production, colors)):
        pair = production_tls_peak_pair(ratio_production, tls_values[index])
        production_pairs[(int(discard), float(coupling))] = pair
        axis.plot(ratio_production, tls_values[index], color=color, lw=1.5, label=f"{coupling:.2f}")
        if pair is not None:
            axis.scatter(
                [pair["left"], pair["right"]],
                [pair["left_height"], pair["right_height"]],
                color=color, edgecolor="k", linewidth=0.35, s=20, zorder=3,
            )
    axis.axvline(1.0, color="0.3", ls="--", lw=0.8)
    axis.set(xlabel=r"$\omega_d/(\Omega/2)$", title=f"discard={discard}")
axes[0].set_ylabel(r"$A_{TLS}^{\perp}$")
axes[-1].legend(title=r"$g$", fontsize=8, ncol=2)
fig.suptitle("Observed TLS-response peaks; dots are extracted extrema", y=1.02)
fig.tight_layout()
plt.show()

split_by_window = np.full((len(windows_production), len(g_production)), np.nan)
midpoint_by_window = np.full_like(split_by_window, np.nan)
for row, discard in enumerate(windows_production):
    for column, coupling in enumerate(g_production):
        pair = production_pairs[(int(discard), float(coupling))]
        if pair is not None:
            split_by_window[row, column] = pair["half_split_ratio"]
            midpoint_by_window[row, column] = pair["midpoint"]

resolved_all_windows = np.isfinite(split_by_window).all(axis=0)
split_mean = np.full(len(g_production), np.nan)
split_std = np.full(len(g_production), np.nan)
midpoint_mean = np.full(len(g_production), np.nan)
midpoint_std = np.full(len(g_production), np.nan)
for column in range(len(g_production)):
    split_valid = split_by_window[np.isfinite(split_by_window[:, column]), column]
    midpoint_valid = midpoint_by_window[np.isfinite(midpoint_by_window[:, column]), column]
    if len(split_valid):
        split_mean[column] = np.mean(split_valid)
        midpoint_mean[column] = np.mean(midpoint_valid)
    if len(split_valid) >= 2:
        split_std[column] = np.std(split_valid, ddof=1)
        midpoint_std[column] = np.std(midpoint_valid, ddof=1)

# Pre-registered acceptance: all three windows resolve the pair, the midpoint
# remains within 0.04 of resonance, and the window-to-window CV is below 10%.
robust_pair = (
    resolved_all_windows
    & (np.abs(midpoint_mean - 1.0) <= 0.04)
    & (split_std / split_mean <= 0.10)
)

fit_coefficients = np.polyfit(g_production[robust_pair], split_mean[robust_pair], 1)
fit_prediction = np.polyval(fit_coefficients, g_production[robust_pair])
fit_r2 = 1.0 - np.sum((split_mean[robust_pair] - fit_prediction) ** 2) / np.sum(
    (split_mean[robust_pair] - np.mean(split_mean[robust_pair])) ** 2
)
omega_half = float(loaded_long_g["metadata_Omega"]) / 2.0
physical_fit_coefficients = fit_coefficients * omega_half

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
markers = ["o", "s", "^"]
for row, (discard, marker) in enumerate(zip(windows_production, markers)):
    axes[0].plot(
        g_production, split_by_window[row], marker=marker, alpha=0.55,
        label=f"discard={discard}",
    )
    axes[1].plot(
        g_production, midpoint_by_window[row], marker=marker, alpha=0.55,
        label=f"discard={discard}",
    )
axes[0].errorbar(
    g_production[robust_pair], split_mean[robust_pair], yerr=split_std[robust_pair],
    fmt="ko", capsize=3, label="accepted mean ± s.d.", zorder=5,
)
dense_g = np.linspace(g_production[robust_pair].min(), g_production[robust_pair].max(), 200)
axes[0].plot(
    dense_g, np.polyval(fit_coefficients, dense_g), "k--",
    label=fr"linear guide, $R^2={fit_r2:.3f}$",
)
axes[0].set(xlabel=r"bare coupling $g$", ylabel="TLS peak half-splitting (ratio units)")
axes[1].errorbar(
    g_production[robust_pair], midpoint_mean[robust_pair], yerr=midpoint_std[robust_pair],
    fmt="ko", capsize=3, label="accepted mean ± s.d.", zorder=5,
)
axes[1].axhline(1.0, color="0.3", ls="--", lw=0.8)
axes[1].set(xlabel=r"bare coupling $g$", ylabel="peak-pair midpoint", ylim=(0.96, 1.05))
for axis in axes:
    axis.legend(fontsize=8)
fig.suptitle("Cross-window stability and resolved-coupling scaling")
fig.tight_layout()
plt.show()

print(f"checkpoint = {long_g_path}")
print(f"completion = {completed.sum()}/{completed.size}")
print("accepted g values =", g_production[robust_pair])
for index, coupling in enumerate(g_production):
    resolved_count = int(np.isfinite(split_by_window[:, index]).sum())
    if resolved_count == len(windows_production):
        cv = split_std[index] / split_mean[index]
        print(
            f"g={coupling:.2f}: resolved {resolved_count}/3, "
            f"half={split_mean[index]:.5f} +/- {split_std[index]:.5f}, "
            f"CV={cv:.3f}, midpoint={midpoint_mean[index]:.5f} +/- {midpoint_std[index]:.5f}, "
            f"accepted={bool(robust_pair[index])}"
        )
    else:
        print(f"g={coupling:.2f}: resolved {resolved_count}/3, accepted=False")
print("normalized half-splitting fit [slope, intercept] =", fit_coefficients)
print("normalized fit R^2 =", fit_r2)
print("frequency-unit fit [slope, intercept] =", physical_fit_coefficients)

if "channel_linear_coefficients" in globals():
    print(
        "N=3 channel pilot frequency-unit slope =",
        float(channel_linear_coefficients[0]),
    )
    print(
        "The N=6 response-peak slope and N=3 channel slope are not quantitatively "
        "identified: system size and extracted objects differ."
    )


## 8. Production result: a resolved TLS-response doublet

The completed (N=6) checkpoint contains all (7\times61=427) requested parameter points and all arrays are finite. The central TLS transverse-response peak is unresolved at weak coupling, becomes a genuine two-peak structure, and eventually reaches the finite frequency window.

The acceptance rule is deliberately stricter than a visual peak pick: both peaks must be independently detected in all three discard windows, the peak-pair midpoint must remain within 0.04 of ωₑ/(Ω/2)=1, and the cross-window coefficient of variation of the half-splitting must not exceed 10%. Exactly four couplings pass: (g=0.06,0.08,0.10,0.12).

For those four points, the mean normalized half-splittings are (0.0567,0.0819,0.1114,0.1418), with only 3.3%--9.0% cross-window variation. A linear guide over this **resolved interval** gives

\[
\frac{\delta\omega_{\rm resp}}{\Omega/2}
=1.424\,g-0.0303,
\qquad R^2=0.998.
\]

This is not a weak-coupling asymptotic law: the nonzero intercept records the finite linewidth/resolution threshold. The midpoint drifts from 1.0009 to 1.0376, so the response becomes slightly asymmetric as coupling grows.

- (g=0.02) is unresolved in all windows; (g=0.04) resolves only in the latest window.
- (g=0.16) is excluded because the upper response branch reaches or leaves the present upper boundary 1.25 and additional structures enter the window.
- The production response splitting and the (N=3) channel-pilot splitting are qualitatively consistent with a coupling-induced branch separation, but their slopes must not be equated: they use different system sizes and different extracted objects.

The defensible claim is therefore a **window-stable, coupling-resolved TLS-response doublet in the finite (N=6) open system**, not yet a production-size Floquet-channel avoided crossing.


## 9. Completed damping and open-size checkpoints

The second production checkpoint scans 31 logarithmically spaced damping rates at fixed (N=6), (g=0.08), and ωₑ/(Ω/2)=1. The same discard windows (8,20,40) are retained, so a nonmonotonic feature is accepted only if it survives all three windows.

The third checkpoint scans (N=3,…,7) at three representative detunings. It uses (g=γ₁=0.08), 80 periods, two samples per drive step, and discard 20, as specified by the generating cell above. This checkpoint predates metadata fields, so these settings are established by the code provenance rather than encoded in the `.npz` file itself.


In [ ]:
gamma_candidates = [
    Path("floquet_tls_N6_gamma_checkpoint.npz"),
    Path("upload/floquet_tls_N6_gamma_checkpoint.npz"),
]
size_candidates = [
    Path("floquet_tls_open_size_checkpoint.npz"),
    Path("upload/floquet_tls_open_size_checkpoint.npz"),
]
gamma_path = next((path for path in gamma_candidates if path.exists()), None)
size_path = next((path for path in size_candidates if path.exists()), None)
if gamma_path is None or size_path is None:
    raise FileNotFoundError("Place both completed checkpoints beside this notebook.")

with np.load(gamma_path, allow_pickle=False) as saved:
    gamma_data = {key: np.asarray(saved[key]) for key in saved.files}
with np.load(size_path, allow_pickle=False) as saved:
    size_data = {key: np.asarray(saved[key]) for key in saved.files}

gamma_values = gamma_data["gamma_values"]
gamma_windows = gamma_data["discard_windows"].astype(int)
if not gamma_data["completed"].all():
    raise RuntimeError("gamma checkpoint is incomplete")
for key, expected in {
    "metadata_N": 6,
    "metadata_periods": 80,
    "metadata_samples_per_step": 4,
    "metadata_g": 0.08,
    "metadata_omega_ratio": 1.0,
}.items():
    if not np.isclose(gamma_data[key].item(), expected):
        raise ValueError(f"{key}: expected {expected}, found {gamma_data[key].item()}")
for discard in gamma_windows:
    for metric in [
        "A_edge", "A_tls_transverse", "phase_tls_minus_edge",
        "tls_emission", "Mpi_edge_strobe",
    ]:
        values = gamma_data[f"{metric}_d{discard:02d}"]
        if values.shape != gamma_values.shape or not np.isfinite(values).all():
            raise ValueError(f"invalid gamma array: {metric}, discard={discard}")

N_production = size_data["N_values"]
size_ratios = size_data["omega_ratios"]
if not size_data["completed"].all():
    raise RuntimeError("size checkpoint is incomplete")
if not np.array_equal(N_production, np.arange(3, 8)):
    raise ValueError("unexpected size grid")
if not np.allclose(size_ratios, [0.945, 1.0, 1.0733333333333333]):
    raise ValueError("unexpected size-scan frequency grid")
for metric in ["A_edge_d20", "A_tls_transverse_d20", "tls_emission_d20"]:
    values = size_data[metric]
    if values.shape != (len(N_production), len(size_ratios)) or not np.isfinite(values).all():
        raise ValueError(f"invalid size array: {metric}")


def log_quadratic_extremum(x_values, y_values, index):
    """Refine an interior extremum with a quadratic in log(gamma)."""
    if index <= 0 or index >= len(x_values) - 1:
        return float(x_values[index]), float(y_values[index])
    local_log_x = np.log(x_values[index - 1:index + 2])
    coefficients = np.polyfit(local_log_x, y_values[index - 1:index + 2], 2)
    log_location = -coefficients[1] / (2.0 * coefficients[0])
    if not (local_log_x[0] <= log_location <= local_log_x[-1]):
        return float(x_values[index]), float(y_values[index])
    return float(np.exp(log_location)), float(np.polyval(coefficients, log_location))


emission_peak_gamma = []
emission_peak_value = []
for discard in gamma_windows:
    emission = gamma_data[f"tls_emission_d{discard:02d}"]
    peak = log_quadratic_extremum(gamma_values, emission, int(np.argmax(emission)))
    emission_peak_gamma.append(peak[0])
    emission_peak_value.append(peak[1])
emission_peak_gamma = np.asarray(emission_peak_gamma)
emission_peak_value = np.asarray(emission_peak_value)

edge_stack = np.vstack([
    gamma_data[f"A_edge_d{discard:02d}"] for discard in gamma_windows
])
mpi_stack = np.vstack([
    gamma_data[f"Mpi_edge_strobe_d{discard:02d}"] for discard in gamma_windows
])
edge_suppressed = np.all(edge_stack < 0.1, axis=0)
mpi_suppressed = np.all(mpi_stack < 0.1, axis=0)
edge_suppression_interval = gamma_values[np.where(edge_suppressed)[0][[0, -1]]]
mpi_suppression_interval = gamma_values[np.where(mpi_suppressed)[0][[0, -1]]]

size_fractional_change_67 = {}
for metric in ["A_edge_d20", "A_tls_transverse_d20", "tls_emission_d20"]:
    values = size_data[metric]
    size_fractional_change_67[metric] = (values[-1] - values[-2]) / values[-2]

fig, axes = plt.subplots(2, 3, figsize=(14, 8.2))
window_colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(gamma_windows)))
gamma_metrics = [
    ("A_edge", r"$A_{edge}$"),
    ("Mpi_edge_strobe", r"$|M_{\pi}^{edge}|$"),
    ("tls_emission", r"$\gamma_1\langle n_d\rangle$"),
]
for column, (metric, ylabel) in enumerate(gamma_metrics):
    axis = axes[0, column]
    for discard, color in zip(gamma_windows, window_colors):
        axis.semilogx(
            gamma_values, gamma_data[f"{metric}_d{discard:02d}"],
            "o-", ms=3, lw=1.4, color=color, label=f"discard={discard}",
        )
    axis.set(xlabel=r"TLS damping $\gamma_1$", ylabel=ylabel)
    axis.legend(fontsize=8)
axes[0, 0].axvspan(*edge_suppression_interval, color="0.5", alpha=0.14, label="all windows < 0.1")
axes[0, 1].axvspan(*mpi_suppression_interval, color="0.5", alpha=0.14)
for peak_gamma, peak_value, color in zip(
    emission_peak_gamma, emission_peak_value, window_colors
):
    axes[0, 2].scatter(peak_gamma, peak_value, s=38, color=color, edgecolor="k", linewidth=0.4, zorder=4)
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_title("Intermediate damping suppresses the edge response")
axes[0, 1].set_title("Stroboscopic pi component shows the same crossover")
axes[0, 2].set_title("TLS loading has a window-stable maximum")

ratio_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(size_ratios)))
size_metrics = [
    ("A_edge_d20", r"$A_{edge}$"),
    ("A_tls_transverse_d20", r"$A_{TLS}^{\perp}$"),
    ("tls_emission_d20", r"$\gamma_1\langle n_d\rangle$"),
]
for column, (metric, ylabel) in enumerate(size_metrics):
    axis = axes[1, column]
    for ratio_index, (ratio, color) in enumerate(zip(size_ratios, ratio_colors)):
        axis.semilogy(
            N_production, size_data[metric][:, ratio_index], "o-",
            color=color, label=fr"$\omega_d/(\Omega/2)={ratio:.3f}$",
        )
    axis.set(xlabel=r"chain size $N$", ylabel=ylabel, xticks=N_production)
    axis.legend(fontsize=8)
axes[1, 0].set_title("Edge amplitude remains detuning-sensitive")
axes[1, 1].set_title("TLS amplitude partly converges by N=7")
axes[1, 2].set_title("Local emission is nearly converged from N=6 to 7")
fig.suptitle("Completed damping and finite-size diagnostics", y=1.01)
fig.tight_layout()
plt.show()

print(f"gamma checkpoint completion = {gamma_data['completed'].sum()}/{gamma_data['completed'].size}")
print(f"size checkpoint completion = {size_data['completed'].sum()}/{size_data['completed'].size}")
for discard, peak_gamma, peak_value in zip(
    gamma_windows, emission_peak_gamma, emission_peak_value
):
    print(
        f"discard={discard}: emission maximum gamma={peak_gamma:.5f}, "
        f"value={peak_value:.6f}"
    )
print(
    "emission-maximum geometric mean gamma =",
    float(np.exp(np.mean(np.log(emission_peak_gamma)))),
)
print("edge all-window suppression interval (A_edge < 0.1) =", edge_suppression_interval)
print("pi-component all-window suppression interval (< 0.1) =", mpi_suppression_interval)
print("A_edge at gamma=3 for discard 8,20,40 =", edge_stack[:, -1])
for metric, changes in size_fractional_change_67.items():
    print(metric, "fractional N=6 to 7 changes =", dict(zip(size_ratios, changes)))


## 10. Production conclusions from damping and size scans

### Damping crossover

The TLS emission is genuinely nonmonotonic in γ₁. Quadratic interpolation in (loggamma_1) places its maximum at

\[
\gamma_{\rm load}=0.0376, 0.0310, 0.0268
\]

for discard windows 8, 20, and 40, respectively. Their geometric mean is (0.0315). The precise peak is therefore not a universal parameter, but the existence and scale of the loading maximum are stable.

The edge response supplies the complementary fast-defect-elimination signature. All three windows satisfy (A_{edge}<0.1) throughout (0.0422\legamma_1\le0.1225), whereas at (gamma_1=3) the three amplitudes recover to (0.683,0.641,0.578). The stroboscopic π component has a similar all-window suppression band, (0.0341\legamma_1\le0.1516), followed by recovery.

Individual zeros or minima of (A_{edge}) occur at different γ₁ for different discard windows. They are finite-window phase/beating features and must not be labeled a phase boundary. The robust statement is the three-regime crossover

\[
\text{coherent slow defect}
\;\longrightarrow\;
\text{intermediate dissipative loading}
\;\longrightarrow\;
\text{fast-defect elimination and edge recovery}.
\]

### Size diagnostic

The (N=6\to7) changes in TLS emission are only (1.2%-1.6%) at all three detunings, so this local dissipative observable is nearly converged. The resonant edge amplitude also changes by only (2.0%), but the two off-resonant edge amplitudes still change by about (29%). The resonant TLS transverse amplitude changes by (13.4%).

Thus the scan supports finite-size convergence of local emission, but it does **not** establish exponential edge-lifetime scaling or a thermodynamic DTC. Obtaining such a claim would require several larger sizes and a lifetime or channel-eigenvalue observable rather than one fixed-time Fourier amplitude.


## 11. Evidence status after the three production checkpoints

The completed numerical evidence now supports three finite-system claims:

1. A window-stable TLS-response doublet is resolved for (0.06\le g\le0.12), with an approximately linear separation over that resolved interval.
2. TLS loading is nonmonotonic in damping, and the edge response recovers in the fast-defect regime rather than degrading monotonically.
3. Local TLS emission is nearly size-converged by (N=6,7), while the coherent edge response remains detuning- and size-sensitive.

It still does not support a thermodynamic many-body-DTC claim, a unique damping phase boundary, or a production-size avoided crossing of Floquet-channel eigenvalues. Those distinctions should remain explicit in the paper.

If one more expensive calculation is performed, the highest-value target is the open-system Floquet π eigenvalue (or a controlled reduced edge--TLS channel) at the same (N,g,gamma_1) points used here, allowing the observed decay lifetime and channel spectrum to be compared directly.
